# Part 4 · Notebook 08 — Kill switch, pre-trade checks and multi-asset

**Sessions:** S13 (Kill switch) · S14 (Futures, FX & crypto) · S15 (Pre-trade checks) · S16 (Contract tests) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Compute futures P&L and FX pip values exactly.
2. Find a futures expiry and its roll date.
3. Write the pre-trade check chain every order passes.
4. Plan a kill-switch flatten, make it sticky, and time it.
5. Run the same contract tests against two broker adapters.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()
from datetime import date, timedelta
import asyncio, tempfile, time

## 1. Futures: the multiplier is the risk

A futures P&L is `(exit − entry) × qty × multiplier`, with `qty` signed (+ long, − short). `p.FUTURES` holds `(tick, multiplier)`.

In [ ]:
pd.DataFrame({k: {"tick": t, "multiplier": m, "tick value $": t * m} for k, (t, m) in p.FUTURES.items()}).T

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def futures_pnl(entry: Decimal, exit: Decimal, qty: Decimal, multiplier: Decimal) -> Decimal:
    return ...                                    # ✍️

D = Decimal
trades = [("ES", D("5012.25"), D("5020.00"), D("2")), ("NQ", D("18010.50"), D("17950.25"), D("1")),
          ("CL", D("71.42"), D("70.95"), D("-3")), ("MES", D("5012.25"), D("5020.00"), D("2")),
          ("GC", D("2350.1"), D("2361.4"), D("-1"))]
mine = [futures_pnl(e, x, q, p.FUTURES[s][1]) for s, e, x, q in trades]
mine = p.check("futures P&L", mine, [p.futures_pnl(e, x, q, p.FUTURES[s][1]) for s, e, x, q in trades])
dict(zip([t[0] for t in trades], mine))

The same 7.75-point move is $775 on 2 ES and $77.50 on 2 MES. Micro contracts exist so a small account can size correctly. FX works the same way through the pip value: one pip (0.0001, or 0.01 for JPY pairs) × units × the USD value of the quote currency.

In [ ]:
fx = [("EURUSD", D("100000"), D("1")), ("USDJPY", D("100000"), D("1") / D("151.20")), ("GBPUSD", D("10000"), D("1"))]
{pair: f"${p.pip_value_usd(pair, units, usd).quantize(D('0.01'))} per pip" for pair, units, usd in fx}

## 2. Expiry and roll

Equity index futures expire on the **third Friday** of the contract month. Volume moves to the next contract about a week earlier, so we roll a fixed number of business days before expiry (8 here, holidays ignored).

`date.weekday()` is 0 for Monday … 4 for Friday.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def third_friday(year: int, month: int) -> date:
    first = date(year, month, 1)
    days_to_friday = ...                          # ✍️ days from `first` to the month's first Friday (0 if it is one)
    return first + timedelta(days=days_to_friday + 14)

months = [(2026, 3), (2026, 6), (2026, 9), (2026, 12), (2027, 1), (2025, 8)]
mine = [p.attempt(third_friday, y, m) for y, m in months]
mine = p.check("third_friday", mine, [p.third_friday(y, m) for y, m in months])
mine

In [ ]:
pd.DataFrame([{"contract": f"ES{p.MONTH_CODES[m - 1]}{str(y)[-1]}", "roll on": p.roll_date(y, m), "expires": p.third_friday(y, m)}
              for y, m in [(2026, 3), (2026, 6), (2026, 9), (2026, 12)]])

## 3. The pre-trade check chain

Every order passes the same checks before it reaches an adapter. Return the **first** failure, or `None`:

1. `'kill switch'` if `ctx.killed`;
2. `'no market data'` if the symbol has no `ctx.last` price;
3. `'price outside band'` if `|price / last − 1| > ctx.band` (a fat-finger guard);
4. `'order too large'` if `qty × price > ctx.max_notional`;
5. `'position limit'` if `|position + signed qty| > ctx.max_position` (signed: + for BUY, − for SELL);
6. `'buying power'` if a BUY costs more than `ctx.buying_power`.

In [ ]:
ctx = p.Context()
print(ctx)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pretrade_check(order: dict, ctx: p.Context) -> str | None:
    if ctx.killed:
        return "kill switch"
    last = ctx.last.get(order["symbol"])
    if last is None:
        return "no market data"
    if ...:                                       # ✍️ price more than ctx.band away from last, relatively
        return "price outside band"
    notional = order["qty"] * order["price"]
    if notional > ctx.max_notional:
        return "order too large"
    signed = order["qty"] if order["side"] == "BUY" else -order["qty"]
    if ...:                                       # ✍️ |current position (0 if none) + signed| above ctx.max_position
        return "position limit"
    if order["side"] == "BUY" and notional > ctx.buying_power:
        return "buying power"
    return None

orders = p.random_orders()
mine = [pretrade_check(o, ctx) for o in orders]
mine = p.check("pretrade_check", mine, [p.pretrade_check(o, ctx) for o in orders])
counts = pd.Series(["passed" if r is None else r for r in mine]).value_counts()
counts.plot.barh(title="1,000 orders through the pre-trade checks"); plt.gca().invert_yaxis(); plt.show()

## 4. The kill switch

When it trips: **cancel every open order first** (so nothing new fills), **then** close every position with a market order. Return the plan: `('cancel', id)` for each open order (sorted), then `('market', symbol, side, qty)` for each non-zero position (sorted by symbol): SELL a long, BUY a short, qty always positive.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def flatten_plan(open_orders: list[str], positions: dict[str, Decimal]) -> list[tuple]:
    plan = [("cancel", oid) for oid in sorted(open_orders)]
    plan += ...                                   # ✍️ ("market", symbol, side, abs(qty)) for each non-zero position
    return plan

open_orders = ["qf-310", "qf-302", "qf-305"]
positions = {"SPY": D("75"), "AAPL": D("-10"), "QQQ": D("0"), "NVDA": D("30")}
mine = p.attempt(flatten_plan, open_orders, positions)
mine = p.check("flatten_plan", mine, p.flatten_plan(open_orders, positions))
mine

**Sticky.** A kill switch that forgets it was tripped when the process restarts is not a kill switch. `p.KillSwitch` writes its state to a file; a new process reads it and stays stopped until a human resets it with the confirmation phrase.

In [ ]:
state_file = Path(tempfile.mkdtemp()) / "killswitch.json"
ks = p.KillSwitch(state_file)
ks.trip("daily loss limit hit: -2.1%")
print("tripped:", ks.tripped)
restarted = p.KillSwitch(state_file)                    # a brand-new process after a crash
print("after restart, still tripped:", restarted.tripped)
print("reset with the wrong phrase:", restarted.reset("ok"))
print("reset with the right phrase:", restarted.reset("I have flattened and reviewed"), "→ tripped:", restarted.tripped)
print("audit:", ks.audit + restarted.audit)

**Fast.** The lesson plan's target is flat on both brokers in under 5 seconds. With 20 open orders and 8 positions per broker and 100 ms per request, doing it one request at a time is too slow; `asyncio.gather` sends them concurrently. (Cancels still finish before the closes start.)

In [ ]:
def brokers():
    pos = {f"S{i}": D("10") for i in range(8)}
    return [p.AsyncSimBroker("ib", 20, pos, latency=0.1), p.AsyncSimBroker("alpaca", 20, pos, latency=0.1)]

async def flatten_sequential(bs):
    for b in bs:
        for oid in list(b.open):
            await b.cancel(oid)
        for s in list(b.positions):
            await b.close(s)

async def flatten_concurrent(bs):
    await asyncio.gather(*(b.cancel(oid) for b in bs for oid in list(b.open)))
    await asyncio.gather(*(b.close(s) for b in bs for s in list(b.positions)))

for name, fn in [("one at a time", flatten_sequential), ("concurrent", flatten_concurrent)]:
    bs = brokers(); t0 = time.perf_counter()
    await fn(bs)
    flat = all(not b.open and not any(b.positions.values()) for b in bs)
    took = time.perf_counter() - t0
    print(f"{name:14s}: {took:5.2f} s, flat = {flat}, {'within' if took < 5 else 'MISSES'} the 5 s target")

## 5. Contract tests

Every adapter (IB, Alpaca, the simulator) must behave the same way to the code above it. A **contract test suite** is one set of tests run against every adapter. `SloppyBroker` works in a quick demo, but the suite finds two problems that would cost money.

In [ ]:
pd.DataFrame({"SimBroker": p.contract_tests(p.SimBroker), "SloppyBroker": p.contract_tests(p.SloppyBroker)}).replace({True: "✔ pass", False: "✘ FAIL"})

`retry_is_idempotent` is the one that matters most: after a timeout you resend the **same** client order id, and a correct adapter returns the existing order instead of buying twice (common mistake #3 in the lesson plan).

## Wrap-up

* Know each contract's multiplier and expiry; roll before the volume leaves.
* One pre-trade chain in front of every adapter; the kill switch is check number one.
* Kill switch: cancel, then close; sticky across restarts; fast enough to matter.
* Contract tests keep every adapter honest.
* Graded version: `labs/part04/week16_safety` (`SimBroker` against the full contract suite, sticky `KillSwitch`) and the Clinic W4 kill drill.